In [ ]:
import pandas as pd
import numpy as np
import sys,os
from tqdm import tqdm
import sys, os
import compass
print("Using COMPASS version:", compass.__version__)

from compass.utils import plot_embed_with_label,plot_performance, score2
from compass.tokenizer import CANCER_CODE, CONCEPT
from compass import PreTrainer, FineTuner, loadcompass #, get_minmal_epoch
from compass.tokenizer import CANCER_CODE, CONCEPT

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from copy import deepcopy

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, LeaveOneOut
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression

%matplotlib inline

def onehot(S):
    assert type(S) == pd.Series, 'Input type should be pd.Series'
    dfd = pd.get_dummies(S, dummy_na=True)
    nanidx = dfd[dfd[np.nan].astype(bool)].index
    dfd.loc[nanidx, :] = np.nan
    dfd = dfd.drop(columns=[np.nan])*1.
    cols = dfd.sum().sort_values(ascending=False).index.tolist()
    dfd = dfd[cols]
    return dfd

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def train_lgr_with_gridsearch(
    train_X, train_y,
    scoring='roc_auc', cv=10, n_jobs=-1, random_state=42):
    """
    1) GridSearchCV find best parameters
    2) best parameters for refit(train_X, train_y) 
    3) return: best_estimator_, best_params_, best_cv_score, final_model
    """
    # Build a pipeline: standardize -> logistic regression
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l2", solver="lbfgs", class_weight="balanced",
            max_iter=1000, random_state=random_state
        ))
    ])

    # Param grid (only search C on the classifier step)
    param_grid = {
        "clf__C": np.logspace(-3, 2, 100)  # 
    }

    # Stratified CV for classification
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    gcv = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=scoring,
        cv=skf,
        n_jobs=n_jobs,
        refit=True,          # refit pipeline on the whole train set using best params
        verbose=0
    )
    gcv.fit(train_X, train_y)

    best_params = gcv.best_params_
    best_cv_score = gcv.best_score_
    # best_estimator = gcv.best_estimator_   # already refit on full train set

    # Optionally, extract best C for reporting
    best_C = best_params["clf__C"]

    print(f"[GridSearch] best_C={best_C:.3g}, best_{scoring}={best_cv_score:.4f}")

    # best_estimator is the final model trained on the full training data
    return gcv

In [ ]:
data_path = './data/ITRP/'
df_label = pd.read_pickle(os.path.join(data_path, 'ITRP.PATIENT.TABLE'))
df_tpm = pd.read_pickle(os.path.join(data_path, 'ITRP.TPM.TABLE'))
df_tpm.shape, df_label.shape

if not os.path.exists("./results"):
    os.makedirs("./results")

dfcx = df_label.cancer_type.map(CANCER_CODE).to_frame('cancer_code').join(df_tpm)

In [ ]:
s = df_label.cohort.value_counts()
large_medium_cohorts = s.index.tolist() #s[s  > 30].index.tolist() #
#[s.index != 'Kim']
#df_label = df_label[df_label.cohort.isin(large_cohorts)]
cohort_list = df_label.cohort.unique().tolist()

In [ ]:
cohort_map = s.index + '(n=' + s.astype(str) + ')'

In [ ]:
y = onehot(df_label.response_label)

In [ ]:
def leave_one_cohort_out(cohorts):
    # Create a list of lists, each missing one element from the original list
    return [(cohorts[i], cohorts[:i] + cohorts[i+1:]) for i in range(len(cohorts))]
train_test_cohorts = leave_one_cohort_out(cohort_list)

## ssGSEA-Avg-LGR

In [ ]:
X1 = pd.read_csv('./data/GSEA/ITRP_avg_concept_43.csv', index_col=0)
X3 = pd.read_csv('./data/GSEA/ITRP_ssGSEA_concept_43.csv', index_col=0)

X5 = pd.read_csv('./data/GSEA/ITRP_avg_signature_132.csv', index_col=0)
X7 = pd.read_csv('./data/GSEA/ITRP_ssGSEA_signature_132.csv', index_col=0)

Modes = ['Geometric mean from Gene TPM', 
         'ssGSEA from Gene TPM', 
        ]
Xs1 = [X1, X3]
Xs2 = [X5, X7]
levels = ['concepts', 'signatures']

final_performance = []

for level, Xs in zip(levels, [Xs1, Xs2]):
    for X, mode in zip(Xs, Modes):
        
        res = []
        for test_cohort, train_cohorts in train_test_cohorts:
    
            train_cohort_name = 'Leave_%s_out' % test_cohort
            ## Get data for this cohort
            cohort_idx = df_label[df_label['cohort'].isin(train_cohorts)].index
            cohort_X = X.loc[cohort_idx]
            cohort_y = y.loc[cohort_idx]
    
            train_X = cohort_X 
            train_y = cohort_y['R'].values
    
            test_cohort_idx = df_label[df_label['cohort'] == test_cohort].index
            test_cohort_X = X.loc[test_cohort_idx]

            test_cohort_y = y.loc[test_cohort_idx]['R']
    
            gcv = train_lgr_with_gridsearch(train_X, train_y)
            test_pred_prob = gcv.best_estimator_.predict_proba(test_cohort_X)
            dfp = pd.DataFrame(test_pred_prob, index =test_cohort_y.index )
    
            dfp['train_cohort'] = train_cohort_name
            dfp['test_cohort'] = test_cohort    
            dfp['best_C'] = gcv.best_params_["clf__C"]
            dfp['mode'] = mode
            
            dfp = dfp.join(test_cohort_y)
            y_true, y_prob, y_pred = dfp['R'], dfp[1], dfp[[0, 1]].idxmax(axis=1)
            
            res.append(dfp)
    
        dfs = pd.concat(res)
        dfp = dfs.groupby(['train_cohort', 'test_cohort']).apply(lambda x:score2(x['R'], x[1], x[[0, 1]].idxmax(axis=1)))
        mode_map = dfs.groupby('train_cohort')['mode'].unique().apply(lambda x:x[0])
        c_map = dfs.groupby('train_cohort')['best_C'].unique().apply(lambda x:x[0])
        
        #roc, prc, f1, acc, mcc
        dfp = dfp.apply(pd.Series)
        dfp.columns = ['ROC', 'PRC', 'F1', 'ACC', 'MCC']
        dfp = dfp.reset_index()
        dfp['mode'] = dfp.train_cohort.map(mode_map)
        dfp['best_C'] = dfp.train_cohort.map(c_map)
        dfp['level'] = level
        final_performance.append(dfp)
dfp1 = pd.concat(final_performance)

## COMPASS-LGR

In [ ]:
seed = 42 # use 24, 42, 64
mode = 'COMPASS'
Modes.append(mode)

levels = ['concepts', 'signatures']

final_performance = []

for level in levels:

    res = []
    for test_cohort, train_cohorts in train_test_cohorts:
    
        pth = f'./compass_run/FT_v100/LOCO_PFT_{seed}/leave_{test_cohort}_out.pt'
        compass_model = loadcompass(pth, map_location = 'cuda:0')
        dfcx1 = dfcx[['cancer_code']].join(dfcx[compass_model.feature_name])
        dfe, dfg, dfc = compass_model.extract(dfcx1,  batch_size= 128, with_gene_level = True)

        if level == 'concepts':
            X = dfc
            X = X[X.columns[1:]]
        else:
            X = dfg
            X = X[X.columns[1:]]
    
        train_cohort_name = 'Leave_%s_out' % test_cohort
        ## Get data for this cohort
        cohort_idx = df_label[df_label['cohort'].isin(train_cohorts)].index
        cohort_X = X.loc[cohort_idx]
        cohort_y = y.loc[cohort_idx]

        train_X = cohort_X 
        train_y = cohort_y['R'].values

        test_cohort_idx = df_label[df_label['cohort'] == test_cohort].index
        test_cohort_X = X.loc[test_cohort_idx]

        test_cohort_y = y.loc[test_cohort_idx]['R']

        gcv = train_lgr_with_gridsearch(train_X, train_y)
        test_pred_prob = gcv.best_estimator_.predict_proba(test_cohort_X)
        dfp = pd.DataFrame(test_pred_prob, index =test_cohort_y.index )

        dfp['train_cohort'] = train_cohort_name
        dfp['test_cohort'] = test_cohort    
        dfp['best_C'] = gcv.best_params_["clf__C"]
        dfp['mode'] = mode
            
        dfp = dfp.join(test_cohort_y)
        y_true, y_prob, y_pred = dfp['R'], dfp[1], dfp[[0, 1]].idxmax(axis=1)
        
        res.append(dfp)

    dfs = pd.concat(res)
    dfp = dfs.groupby(['train_cohort', 'test_cohort']).apply(lambda x:score2(x['R'], x[1], x[[0, 1]].idxmax(axis=1)))
    mode_map = dfs.groupby('train_cohort')['mode'].unique().apply(lambda x:x[0])
    c_map = dfs.groupby('train_cohort')['best_C'].unique().apply(lambda x:x[0])
    
    #roc, prc, f1, acc, mcc
    dfp = dfp.apply(pd.Series)
    dfp.columns = ['ROC', 'PRC', 'F1', 'ACC', 'MCC']
    dfp = dfp.reset_index()
    dfp['mode'] = dfp.train_cohort.map(mode_map)
    dfp['best_C'] = dfp.train_cohort.map(c_map)
    dfp['level'] = level
    final_performance.append(dfp)
dfp2 = pd.concat(final_performance)

In [ ]:
dfp = dfp1._append(dfp2)
dfpf = dfp[dfp.test_cohort.isin(large_medium_cohorts)]

In [ ]:
roc = dfpf.groupby(['level','mode', 'test_cohort']).ROC.mean().unstack().T
roc = roc.loc[large_medium_cohorts].round(3)
roc.index = roc.index.map(cohort_map)
roc_mean = roc.mean().to_frame(name='Mean').T
roc_std = roc.std().to_frame(name='Mean').T
roc_mean = roc_mean.round(3).astype(str) + '±' + roc_std.round(3).astype(str)
roc = roc.round(3).astype(str)._append(roc_mean)
roc.to_excel('./results/COMPASS_GLM_ssGSEA_ROC.xlsx')
roc

In [ ]:
prc = dfpf.groupby(['level','mode', 'test_cohort']).PRC.mean().unstack().T
prc = prc.loc[large_medium_cohorts].round(3)
prc.index = prc.index.map(cohort_map)
prc_mean = prc.mean().to_frame(name='Mean').T
prc_std = prc.std().to_frame(name='Mean').T
prc_mean = prc_mean.round(3).astype(str) + '±' + prc_std.round(3).astype(str)
prc = prc.round(3).astype(str)._append(prc_mean)
prc.to_excel('./results/COMPASS_GLM_ssGSEA_PRC.xlsx')
prc

In [ ]:
palette = ['#F6C6AD', '#E59EDD', '#61CBF4']

In [ ]:
from statannotations.Annotator import Annotator

x='level'
y='ROC'
hue = 'mode'
labels =  ['TPM-Average-LGR',
           'TPM-ssGSEA-LGR',
          'TPM-COMPASS-LGR',      
          ]

order = ['signatures', 'concepts']
x_labelticks = ['Signature level\n(dim=132)', 'Concept level\n(dim=43)']
width = 0.5

fig, ax = plt.subplots(figsize=(6, 4.5), )
sns.barplot(data  = dfpf, x = x, errorbar = 'se', hue= hue,order = order,saturation = 1,
            hue_order = Modes, capsize=0.1, width = width, palette=palette,
            #legend=False,
            y = y, ax=ax)

ax.set_ylim(0.5, 0.8)
ax.set_ylabel('AUROC')
ax.set_xlabel('')
sns.despine()

ax.tick_params(bottom=True, left=True)


handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, labels,
           title="Logistic Regression", 
           loc='center left',        
           bbox_to_anchor=(1.0, 0.5), 
           frameon=False)

ax.set_xticklabels(x_labelticks, rotation = 0)

pairs = [
         (('concepts', 'ssGSEA from Gene TPM'),
          ('concepts','COMPASS')),
         (('concepts', 'Geometric mean from Gene TPM'),
          ('concepts','COMPASS')),

         
         (('signatures', 'ssGSEA from Gene TPM'),
          ('signatures','COMPASS')),
         (('signatures', 'Geometric mean from Gene TPM'),
          ('signatures','COMPASS')),
        ]

annot = Annotator(
    ax, pairs,
    data=dfpf, x = x, y = y, hue = hue,
    order = order, hue_order = Modes, width = width, 
    # DO NOT pass plot='barplot'
)
annot.configure(
    test='Wilcoxon',                 # or 'Mann-Whitney'
    #comparisons_correction='fdr_bh',
    text_format='star',
    show_test_name=False
)
annot.apply_and_annotate()

fig.savefig(f'./results/LGR_ROC_barplot.svg',
            bbox_inches = 'tight')

In [ ]:
y='PRC'
fig, ax = plt.subplots(figsize=(6, 4.5), )
sns.barplot(data  = dfpf, x = x, errorbar = 'se', hue= hue,order = order,
            hue_order = Modes, capsize=0.1, width = width, palette=palette,saturation = 1,
            #legend=False,
            y = y, ax=ax)

ax.set_ylim(0.2, 0.75)
ax.set_ylabel('PR-AUC')
ax.set_xlabel('')
sns.despine()

ax.tick_params(bottom=True, left=True)


handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, labels,
           title="Logistic Regression", 
           loc='center left',        
           bbox_to_anchor=(1.0, 0.5), 
           frameon=False)

ax.set_xticklabels(x_labelticks, rotation = 0)


annot = Annotator(
    ax, pairs,
    data=dfpf, x = x, y = y, hue = hue,
    order = order, hue_order = Modes, width = width, 
    # DO NOT pass plot='barplot'
)
annot.configure(
    test='Wilcoxon',  
    text_format='star',
    show_test_name=True
)
annot.apply_and_annotate()


fig.savefig(f'./results/LGR_PRC_barplot.svg',
            bbox_inches = 'tight')